## Week 2 Day 3

Now we get to more detail:

1. Different models

2. Structured Outputs

3. Guardrails

In [12]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool,ModelSettings, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput
import os
from pydantic import BaseModel, Field

In [2]:
load_dotenv(override=True)

True

In [3]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if  openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AQ
OpenRouter API Key exists and begins sk-or-
Groq API Key exists and begins gsk_


In [4]:
instructions = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write compelling sales emails that are likely to get a response.
"""

### It's easy to use any models with OpenAI compatible endpoints in 3 steps:

STEP 1: Find the OpenAI compatible base URL (see Guide 9 in the guides folder)

In [5]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

STEP 2: Create a python client library instance (the async version)

In [6]:
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)


STEP 3: Create a model object

In [7]:
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite", openai_client=gemini_client)
# kimi_model = OpenAIChatCompletionsModel(model="moonshotai/kimi-k2-instruct", openai_client=groq_client)
kimi_model = OpenAIChatCompletionsModel(model="moonshotai/kimi-k2.6", openai_client=openrouter_client, 
                                         )
oss_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)

In [8]:
# use it with kimi as its a trillion params model
model_config = ModelSettings(
    extra_body={"max_tokens": 6000} 
)

In [ ]:
sales_agent1 = Agent(name="Gemini Sales Agent", instructions=instructions, model=gemini_model)
sales_agent2 = Agent(name="Kimi2 Sales Agent", instructions=instructions, model=kimi_model, model_settings=model_config )
sales_agent3 = Agent(name="GPT-OSS Sales Agent",instructions=instructions, model=oss_model)

In [10]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [30]:
from pathlib import Path
import importlib.util

# Load the repository messenger module by file path to avoid stale or conflicting imports.
notebook_dir = Path.cwd()
messenger_locations = [
    notebook_dir / "messenger.py",
    notebook_dir / "2_openai" / "messenger.py",
    notebook_dir.parent / "messenger.py",
]

for messenger_path in messenger_locations:
    if messenger_path.is_file():
        break
else:
    raise ModuleNotFoundError("Could not locate 2_openai/messenger.py")

spec = importlib.util.spec_from_file_location("project_messenger", messenger_path)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load {messenger_path}")
project_messenger = importlib.util.module_from_spec(spec)
spec.loader.exec_module(project_messenger)
send_email_extra = project_messenger.send_email_extra
push = project_messenger.push

USE_EMAIL = True

def send_message(name, subject, text_body, html_body, signature):
    if USE_EMAIL:
        send_email_extra(name, subject, text_body, html_body, signature)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

In [ ]:
# send_message("Yet another test", "Hooray!", "<html><body><h1>Hooray!</h1></body></html>")

In [16]:
@function_tool
def send_email_tool(name : str, subject: str, text_body: str, html_body: str, signature: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        name: The name of the recipient
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
        signature: The signature to append to the email
    """
    send_message(name, subject, text_body, html_body, signature)
    return "Email sent successfully"

In [22]:
tools = [tool1,tool2,tool3]

In [ ]:
class SendEmail(BaseModel):
    name: str = Field(description="The user's display name")
    subject: str = Field(description="The email subject")
    text_body: str = Field(description="The plain-text email body")
    html_body: str = Field(description="The HTML email body")
    signature: str = Field(description="The signature to append to the email")

In [43]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_agent tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3.Send agent name in email signature.

4. Give user a slang cowboy name
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, 
                       output_type=SendEmail, tools=tools, model=kimi_model, model_settings=model_config)

In [44]:
with trace("Sales Manager across different models"):
    result = await Runner.run(sales_manager, task)
print(result.final_output)
email_content = result.final_output
send_message(email_content.name, email_content.subject, email_content.text_body, email_content.html_body, email_content.signature)

name='ComplAI' subject='SOC2 readiness without the sprint?' text_body="Hi [First Name],\n\nI'm reaching out because SOC2 prep usually forces engineering teams to choose between shipping product and chasing down evidence.\n\nComplAI automates the bulk of compliance work—continuous control monitoring, evidence collection, and gap detection—so you can get audit-ready without halting your roadmap. Most customers cut their prep time by more than half.\n\nWorth a quick 10-minute chat to see if it fits [Company]'s timeline?\n\nBest," html_body='<p>Hi [First Name],</p>\n<p>I\'m reaching out because SOC2 prep usually forces engineering teams to choose between shipping product and chasing down evidence.</p>\n<p>ComplAI automates the bulk of compliance work—continuous control monitoring, evidence collection, and gap detection—so you can get audit-ready without halting your roadmap. Most customers cut their prep time by more than half.</p>\n<p>Worth a quick <a href="[Calendar Link]">10-minute chat

In [38]:
email_content = result.final_output
send_message(email_content.name, email_content.subject, email_content.text_body, email_content.html_body, email_content.signature)

## Check out the trace

https://platform.openai.com/traces

## Part 2: Structured Outputs

An LLM produces text in natural language. But we can have it instead produce a "python object".

This is accomplished using the usual trickery: clever prompts & json!

1. We specify a Python object  
2. In the System prompt, the LLM is instructed to respond in JSON and follow a Schema which represents the Python object  
3. The LLM outputs JSON, and the framework populates a Python object based on it

When we specify the Python object, we create a subclass of BaseModel, which is part of the Pydantic framework.

Pydantic is a framework that easily allows defining a JSON schema and mapping between Python and json.

NOTES:

1. There is something about the way this is done that IS really clever - if you're interested, look up "constrained decoding".
2. Not all providers support Structured Outputs.


In [15]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

In [16]:
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the email is professional and appropriate',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': 'The number of sentences in the body of the email, not including the greeting and signature',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Whether the email contains placeholders for personalization',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [17]:
email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.

Laters.

Ed
"""

In [ ]:
checker = Agent(name="Checker", instructions="You review potential sales emails", model="gpt-5.4-mini", output_type=EmailReview)
# result = await Runner.run(checker, email)

# oss_model from gpt is failing here
# model_settings=model_config, for kimi

In [127]:
review = result.final_output
review

EmailReview(is_professional=False, number_of_sentences=3, contains_placeholders=True)

In [18]:
review.is_professional

NameError: name 'review' is not defined

## Part 3: Guardrails

Guardrails are extremely important in AgenticAI. Put simply, they are controls that you code either in logic or with another LLM call, to prevent undesirable behavior.

For me, the Guardrails impementation in OpenAI Agents SDK feels a bit like "framework voodoo". I suspect their motivation was to show framework-level controls to address this important topic.

But it's simple and clean to implement guardrails explicitly, as separate Runner.run() calls, or checks in your tool implementations.

Regardless - let's take a look at the framework tooling.

https://openai.github.io/openai-agents-python/guardrails/


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Watch out for a Gotcha</h2>
            <span style="color:#ff7800;">There are 3 types of Guardrails in OpenAI Agents SDK: input, output and tool. Input guardrails only run for the first input to the first Agent in Runner.run(). Output guardrails only run for the final output of the last agent. If you have guardrails on other agents, they will never be called.
            </span>
        </td>
    </tr>
</table>

In [19]:
@output_guardrail
async def email_guardrail(ctx, agent, message):
    print(f"Guardrail checking email: {ctx.context}, {agent.name}, {message}")
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(output_info={"review": review},tripwire_triggered=is_problem)

In [20]:
cowboy_instructions = instructions + "\nSpeak like a cowboy"

sales_agent_cowboy = Agent(name="Cowboy", instructions=cowboy_instructions, model=kimi_model,
                           model_settings=model_config, output_guardrails=[email_guardrail])

In [22]:
result = await Runner.run(sales_agent_cowboy, "Write a cold sales email")
result.final_output

Guardrail checking email: None, Cowboy,  Well howdy, partner! 

Now, I reckon you're lookin' for the sharpest, straightest-shootin' cold sales email this side of the Mississippi. I've been wranglin' leads at ComplAI for a good spell, and here's the one that brings 'em in like cattle to a waterin' hole:

---

**Subject:** Herding compliance doesn't have to be a rodeo

Howdy [First Name],

I'll cut straight to the chase—compliance audits and regulatory wranglin' can feel like trying to herd cattle through a dust storm. Messy. Unpredictable. And likely to cost you a fortune if one gets loose.

That's exactly why we built ComplAI.

We work with [Industry] teams like yours to lasso all that compliance chaos into one corral—automating the busywork, flagging risks before they bolt, and cutting audit prep time down to a brisk trot. Most folks we partner with save [X] hours a month and sleep better knowin' the regulators won't be knockin' down their barn doors.

Worth a quick 15-minute chat nex

NameError: name 'checker' is not defined

Check out the trace:

https://platform.openai.com/traces

## On the other hand..

To state the obvious, this is simpler and will work in any framework

In [21]:
simple_cowboy = Agent(name="Simple Cowboy", instructions=cowboy_instructions, model=kimi_model, model_settings=model_config, tools=[send_email_tool], output_guardrails=[email_guardrail])
result = await Runner.run(simple_cowboy, "Write a cold sales email")
email = result.final_output
print(email)


Guardrail checking email: None, Simple Cowboy,  Well howdy, partner! You’ve come to the right wrangler. In my years ridin’ the sales range, I’ve learned the best cold email is like a good lasso: short, accurate, and it hits the mark before the prospect even knows what happened.

Now, I’d usually want to know exactly what ComplAI is fixin’ before I send this to the whole herd, but here’s a rip-roarin’ template that works for most AI-powered compliance outfits. It’s straight-shootin’, respects the prospect’s time, and asks for a low-stakes chat.

**Subject:** quick question about compliance at {{Company}}

**Email:**

Hi {{FirstName}},

Most {{Job Title}}s I talk to are spending 10+ hours a week wrangling paperwork and compliance reviews instead of running the ranch.

ComplAI uses AI to automate the grunt work—cutting review time by 60% without adding headcount.

Worth a brief conversation to see if it fits your operation next Tuesday?

Best,  
{{Your Name}}

---

**Why this works:**
- *

NameError: name 'checker' is not defined

In [154]:
result = await Runner.run(checker, email)
review = result.final_output
if not review.is_professional or review.contains_placeholders:
    print("The email is not professional or has placeholders and will not be sent")
else:
    print("Email is good")

Email is good


## Check out the trace:

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• Try different models<br/>• Add more input and output guardrails<br/>• Use structured outputs for the email generation
            </span>
        </td>
    </tr>
</table>

## OPTIONAL EXTRA: Sandbox Agents

This example will only work on Windows + WSL2, or Mac, or Linux

https://openai.github.io/openai-agents-python/sandbox_agents/

This is an execution harness - a runtime - "a persistent workspace where it can search large document sets, edit files, run commands, generate artifacts, and pick work back up from saved sandbox state."

You have to set up:
1. Manifest: the workspace
2. Capabilities: what it can do
3. SandboxRunConfig: where it runs

In [25]:
from pathlib import Path

import docker

from agents.run import RunConfig

from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig

from agents.sandbox.capabilities import Capabilities

from agents.sandbox.entries import LocalDir

from agents.sandbox.sandboxes.docker import DockerSandboxClient, DockerSandboxClientOptions


In [34]:
from pathlib import Path

# Resolve paths whether the notebook kernel starts at the repository root or 2_openai.
notebook_cwd = Path.cwd()
if (notebook_cwd / "2_openai" / "code").is_dir():
    openai_dir = notebook_cwd / "2_openai"
elif (notebook_cwd / "code").is_dir():
    openai_dir = notebook_cwd
else:
    raise FileNotFoundError(f"Could not find 2_openai/code from {notebook_cwd}")

CODE_DIR = (openai_dir / "code").resolve()
OUTPUT_DIR = (openai_dir / "output").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Code directory: {CODE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Code files: {[path.name for path in CODE_DIR.iterdir()]}")

Code directory: D:\ArtificialIntelligence\agents\2_openai\code
Output directory: D:\ArtificialIntelligence\agents\2_openai\output
Code files: ['orders.py']


In [26]:
CODE_DIR

WindowsPath('D:/ArtificialIntelligence/agents/2_openai/code')

In [39]:
instructions = """
You are a software engineer that fixes bugs.

Review the Python files in the sandbox workspace directory /workspace/code.
Inspect the code, identify the bug, and fix it.
Use workspace-relative paths with the sandbox tools: read code/orders.py and write the corrected file to output/orders.py.
Do not only describe the fix: create the corrected file in the output directory.
Do not use absolute paths in apply_patch operations.

Respond with a summary of what you did.
"""

In [30]:
manifest = Manifest(entries={"code": LocalDir(src=CODE_DIR), "output": LocalDir(src=OUTPUT_DIR)})

capabilities = Capabilities.default()

capabilities

# manifest


[Filesystem(type='filesystem', session=None, run_as=None, configure_tools=None),
 Shell(type='shell', session=None, run_as=None, configure_tools=None),
 Compaction(type='compaction', session=None, run_as=None, policy=None)]

In [31]:
run_config = RunConfig(sandbox=SandboxRunConfig(client=DockerSandboxClient(docker.from_env()), options=DockerSandboxClientOptions(image="python:3.14-slim")), workflow_name="Sandbox coding example")

In [40]:
agent = SandboxAgent(
    name="Engineer",
    instructions=instructions,
    model="gpt-5.4-mini",
    default_manifest=manifest,
    capabilities=capabilities,
)

In [42]:
result = await Runner.run(
    agent,
    "Inspect code/orders.py, fix its bug, and write the corrected file to output/orders.py using workspace-relative paths.",
    run_config=run_config,
)
print(result.final_output)
print("Output files:", [path.name for path in OUTPUT_DIR.iterdir()])

I checked `code/orders.py`, but the workspace appears empty: the file read returned no contents, and a full file search under `/workspace` found nothing to copy or fix.

- I wasn’t able to identify the bug because there’s no accessible source in `code/orders.py`.
- I also couldn’t write `output/orders.py` without the original file contents.

If you can provide the file contents or mount the repo correctly, I can patch it immediately and write the corrected version to `output/orders.py`.
Output files: []


In [45]:
@function_tool
def read_code_file(filename: str) -> str:
    """Read one source file from the local code directory."""
    path = (CODE_DIR / filename).resolve()
    if path.parent != CODE_DIR or not path.is_file():
        raise ValueError(f"Source file is not available: {filename}")
    return path.read_text(encoding="utf-8")


@function_tool
def write_output_file(filename: str, content: str) -> str:
    """Write one corrected source file to the local output directory."""
    path = (OUTPUT_DIR / filename).resolve()
    if path.parent != OUTPUT_DIR:
        raise ValueError(f"Invalid output filename: {filename}")
    path.write_text(content, encoding="utf-8")
    return f"Wrote {path.name} to {OUTPUT_DIR}"


groq_coding_agent = Agent(
    name="Groq Coding Agent",
    instructions=(
        "You fix bugs in Python source files. "
        "Use read_code_file to inspect orders.py, identify the bug, and then use "
        "write_output_file to save the complete corrected source as orders.py. "
        "Do not merely describe the fix."
    ),
    model=oss_model,
    tools=[read_code_file, write_output_file],
)

In [46]:
result = await Runner.run(
    groq_coding_agent,
    "Fix the bug in orders.py and save the complete corrected file as orders.py.",
)
print(result.final_output)
print("Output files:", [path.name for path in OUTPUT_DIR.iterdir()])

The bug was due to using `dict.fromkeys(customer_ids, [])`, which creates a single shared list for all keys, causing all customers to share the same order list. 

The corrected `orders.py` now:
- Imports `List` and `Dict` for proper type hints.
- Uses a dictionary comprehension to create a distinct list for each `customer_id`.
- Provides detailed docstring explaining the fix.

```python
from pydantic import BaseModel
from typing import List, Dict


class Order(BaseModel):
    customer_id: str
    item: str
    total_price: float
    status: str


def group_orders_by_customer_id(orders: List[Order]) -> Dict[str, List[Order]]:
    """Group orders by ``customer_id``.

    Returns a dictionary where each key is a ``customer_id`` and the value is a
    list of ``Order`` objects belonging to that customer.

    The previous implementation used ``dict.fromkeys(customer_ids, [])`` which
    creates a *single* list instance shared by all keys. As a result, appending
    an order to one customer

## OPTIONAL EXTRA: MCP Teaser!

In [47]:
from agents.mcp import MCPServerStreamableHttp


In [48]:
task = """
In the new SandboxAgents feature in the OpenAI Agents SDK as of May 2026, what is the role of the Manifest object?
Always be accurate. If you don't know the answer, say so.
"""

In [49]:
agent = Agent(name="Expert", instructions="Answer the question", model=oss_model)
result = await Runner.run(agent, task)
print(result.final_output)

I’m not familiar with the details of the SandboxAgents feature or the Manifest object in the OpenAI Agents SDK as of May 2026. My training data only goes up to mid‑2024, and I don’t have information on that release.


In [53]:
params = {"url": "https://mcp.context7.com/mcp", "timeout": 60}


async with MCPServerStreamableHttp(name="Context7", params=params) as server:
    agent = Agent(name="Expert", instructions="Use Context7 to answer the question", mcp_servers=[server], model=gemini_model, model_settings=model_config)
    result = await Runner.run(agent, task)

print(result.final_output)

In the OpenAI Agents SDK, the `Manifest` object serves as a configuration structure that defines the environment, security, and filesystem access permissions for a `SandboxAgent`. Its primary roles include:

*   **Defining Environment and Resources:** It specifies the configuration for new sandbox sessions, including the available directory structure, mounted files, and the user identities authorized to operate within the environment.
*   **Controlling Permissions:** By using `Manifest` entries (such as `Dir` or `LocalDir`) along with `Permissions`, developers can define strict access controls—such as read, write, or execute privileges—for specific sandbox users. This ensures that agents operate within a constrained, secure environment.
*   **Managing User Context:** It allows for the declaration of specific users (via the `users` attribute) so that sandbox actions—such as file reads, shell commands, or patches performed by the model—can be mapped to these identities. These permissions

And see the traces:

https://platform.openai.com/traces
